# Two-step SVD for Appendix D

This notebook is split into small sections.
Each section has one short note and one code cell.


## 1. Setup

Define symbols, assumptions, and a helper for normalization.


In [1]:
import sympy as sp

sp.init_printing(use_unicode=False)

lam = sp.symbols('lam', positive=True, real=True)
mu = sp.symbols('mu', positive=True, real=True)
b = sp.symbols('b', positive=True, real=True)
c = sp.symbols('c', nonnegative=True, real=True)

def normalize(v):
    norm_sq = sp.simplify((v.T * v)[0])
    return sp.simplify(v / sp.sqrt(norm_sq))

print('1. Assumptions')
print('Base assumptions: lam > 0, mu > 0, b > 0, c >= 0')
print('For the general c case, we also use b > c so that Sigma is positive definite.')
print('First truncation: keep all directions')
print('Second truncation: keep all 3 macro directions')


1. Assumptions
Base assumptions: lam > 0, mu > 0, b > 0, c >= 0
For the general c case, we also use b > c so that Sigma is positive definite.
First truncation: keep all directions
Second truncation: keep all 3 macro directions


## 2. Input Matrices at c = 0

Build `A`, `Sigma`, `K = Sigma^{-1}` at `c = 0`, and `M = A^T K A`.


In [2]:
A = sp.Matrix([[lam, 0, 0], [0, mu, lam**2 - mu], [0, 0, lam**2]])
Sigma = sp.Matrix([[b, 0, c], [0, b, 0], [c, 0, b]])
Sigma_c0 = sp.simplify(Sigma.subs(c, 0))
K = sp.simplify(Sigma_c0.inv())
M = sp.simplify(A.T * K * A)

print('2. Input matrices at c = 0')
print('A =')
sp.pprint(A)
print('\nSigma at c = 0 =')
sp.pprint(Sigma_c0)
print('\nK = Sigma^{-1} at c = 0 =')
sp.pprint(K)
print('\nM = A^T K A =')
sp.pprint(M)


2. Input matrices at c = 0
A =
[lam  0       0    ]
[                  ]
[            2     ]
[ 0   mu  lam  - mu]
[                  ]
[              2   ]
[ 0   0     lam    ]

Sigma at c = 0 =
[b  0  0]
[       ]
[0  b  0]
[       ]
[0  0  b]

K = Sigma^{-1} at c = 0 =
[1      ]
[-  0  0]
[b      ]
[       ]
[   1   ]
[0  -  0]
[   b   ]
[       ]
[      1]
[0  0  -]
[      b]

M = A^T K A =
[   2                                     ]
[lam                                      ]
[----        0                  0         ]
[ b                                       ]
[                                         ]
[             2             /   2     \   ]
[           mu           mu*\lam  - mu/   ]
[ 0         ---          --------------   ]
[            b                 b          ]
[                                         ]
[                                        2]
[         /   2     \     4   /   2     \ ]
[      mu*\lam  - mu/  lam  + \lam  - mu/ ]
[ 0    --------------  -------

## 3. First-Step Singular Values at c = 0

Compute the singular values of `K` and `M`.


In [3]:
r = sp.sqrt(lam**4 + mu**2)
s1 = sp.simplify(lam**2 / b)
s2 = sp.simplify((lam**4 - lam**2 * mu + mu**2 - (lam**2 - mu) * r) / b)
s3 = sp.simplify((lam**4 - lam**2 * mu + mu**2 + (lam**2 - mu) * r) / b)
sv_M = [s1, s2, s3]
sv_K = [sp.simplify(1 / b), sp.simplify(1 / b), sp.simplify(1 / b)]

print('3. First-step singular values at c = 0')
print('Singular values of K =')
sp.pprint(sv_K)
print('\nSingular values of M =')
sp.pprint(sv_M)


3. First-step singular values at c = 0
Singular values of K =
 1  1  1 
[-, -, -]
 b  b  b 

Singular values of M =
                                               ____________                    >
    2     4      2        2   /     2     \   /    4     2      4      2       >
 lam   lam  - lam *mu + mu  + \- lam  + mu/*\/  lam  + mu    lam  - lam *mu +  >
[----, ----------------------------------------------------, ----------------- >
  b                             b                                              >

>                      ____________ 
>   2   /   2     \   /    4     2  
> mu  + \lam  - mu/*\/  lam  + mu   
> ---------------------------------]
>        b                          


## 4. First-Step Singular Vectors at c = 0

Choose one valid orthonormal basis for the first SVD of `M`.
For `K = (1/b) I`, the basis is not unique, so we use `U_K = I`.


In [4]:
w1 = sp.Matrix([1, 0, 0])
v2 = sp.Matrix([0, -(lam**2 + r), mu])
v3 = sp.Matrix([0, -lam**2 + r, mu])
w2 = normalize(v2)
w3 = normalize(v3)
U_M = sp.Matrix.hstack(w1, w2, w3)
S_M = sp.diag(*sv_M)
U_K = sp.eye(3)
S_K = sp.diag(*sv_K)

print('4. First-step singular vectors at c = 0')
print('U_M =')
sp.pprint(U_M)
print('\nU_K =')
sp.pprint(U_K)


4. First-step singular vectors at c = 0
U_M =
[1                    0                                       0                >
[                                                                              >
[                      ____________                            ____________    >
[               2     /    4     2                      2     /    4     2     >
[          - lam  - \/  lam  + mu                  - lam  + \/  lam  + mu      >
[0  --------------------------------------  ---------------------------------- >
[        _________________________________       _____________________________ >
[       /                               2       /                              >
[      /        /          ____________\       /        /          ___________ >
[     /     2   |   2     /    4     2 |      /     2   |   2     /    4     2 >
[   \/    mu  + \lam  + \/  lam  + mu  /    \/    mu  + \lam  - \/  lam  + mu  >
[                                                              

## 5. Merged Matrix Before Step Two

Appendix D merges the kept directions into one matrix.
Here we keep all directions, so the merged matrix is `[U_M S_M, U_K S_K]`.


In [5]:
second_step_matrix = sp.Matrix.hstack(U_M * S_M, U_K * S_K)

print('5. Merged matrix before the second SVD')
sp.pprint(second_step_matrix)


5. Merged matrix before the second SVD
[   2                                                                          >
[lam                                                                           >
[----                                          0                               >
[ b                                                                            >
[                                                                              >
[      /            ____________\ /                                        ___ >
[      |     2     /    4     2 | |   4      2        2   /     2     \   /    >
[      \- lam  - \/  lam  + mu  /*\lam  - lam *mu + mu  + \- lam  + mu/*\/  la >
[ 0    ----------------------------------------------------------------------- >
[                                 _________________________________            >
[                                /                               2             >
[                               /        /          ____________\     

## 6. Gram Matrix of Step Two

The second-step singular values can be read from `second_step_matrix * second_step_matrix^T`.


In [6]:
second_gram = sp.simplify(second_step_matrix * second_step_matrix.T)
second_gram_simple = sp.simplify(M**2 + sp.eye(3) / b**2)

print('6. Gram matrix of the second-step matrix')
print('second_gram =')
sp.pprint(second_gram)
print('\nSimplified form = M^2 + (1/b^2) I =')
sp.pprint(second_gram_simple)


6. Gram matrix of the second-step matrix
second_gram =
[   4                                                                          >
[lam  + 1                                                                      >
[--------                      0                                               >
[    2                                                                         >
[   b                                                                          >
[                                                                              >
[                 4   2        2   3       4                       /   6       >
[              lam *mu  - 2*lam *mu  + 2*mu  + 1              2*mu*\lam  - 2*l >
[   0          ---------------------------------              ---------------- >
[                              2                                               >
[                             b                                                >
[                                                     

## 7. Second-Step Singular Values at c = 0

Since all directions are kept, the second-step singular values are `sqrt(s_i^2 + 1/b^2)`.


In [7]:
sv_step2 = [sp.simplify(sp.sqrt(s**2 + b**-2)) for s in sv_M]

print('7. Singular values of the second-step matrix at c = 0')
sp.pprint(sv_step2)


7. Singular values of the second-step matrix at c = 0
                     _________________________________________________________ >
                    /                                                     2    >
    __________     /  /                                      ____________\     >
   /    4         /   |   4      2        2   /   2     \   /    4     2 |     >
 \/  lam  + 1   \/    \lam  - lam *mu + mu  - \lam  - mu/*\/  lam  + mu  /  +  >
[-------------, -------------------------------------------------------------- >
       b                                       b                               >

> __       ___________________________________________________________ 
>         /                                                     2      
>        /  /                                      ____________\       
>       /   |   4      2        2   /   2     \   /    4     2 |       
> 1   \/    \lam  - lam *mu + mu  + \lam  - mu/*\/  lam  + mu  /  + 1  
> --, ------------

## 8. Coarse-Graining Matrix W at c = 0

With all 3 macro directions kept, `W` can be taken as the left singular-vector matrix of the second-step matrix.


In [8]:
W = U_M

print('8. Coarse-graining matrix W at c = 0')
sp.pprint(W)


8. Coarse-graining matrix W at c = 0
[1                    0                                       0                >
[                                                                              >
[                      ____________                            ____________    >
[               2     /    4     2                      2     /    4     2     >
[          - lam  - \/  lam  + mu                  - lam  + \/  lam  + mu      >
[0  --------------------------------------  ---------------------------------- >
[        _________________________________       _____________________________ >
[       /                               2       /                              >
[      /        /          ____________\       /        /          ___________ >
[     /     2   |   2     /    4     2 |      /     2   |   2     /    4     2 >
[   \/    mu  + \lam  + \/  lam  + mu  /    \/    mu  + \lam  - \/  lam  + mu  >
[                                                                       

## 9. Check at c = 0

Check that `W` diagonalizes the Gram matrix from the second step.


In [9]:
W_check = sp.simplify(W.T * second_gram * W)

print('9. Check at c = 0')
sp.pprint(W_check)


9. Check at c = 0
[   4                                                                          >
[lam  + 1                                                                      >
[--------                                                                      >
[    2                                                                         >
[   b                                                                          >
[                                                                              >
[                                              ____________                    >
[             8   2        6   3      6   2   /    4     2         4   4       >
[          lam *mu  - 2*lam *mu  - lam *mu *\/  lam  + mu   + 3*lam *mu  + 2*l >
[   0      ------------------------------------------------------------------- >
[                                                                              >
[                                                                              >
[         

## 10. General c: K and Its Singular Values

Now try the general case `c != 0` under `b > c >= 0`.
For `K = Sigma^{-1}`, the singular values stay simple.


In [10]:
K_general = sp.simplify(Sigma.inv())
sv_K_general = [sp.simplify(1 / (b - c)), sp.simplify(1 / b), sp.simplify(1 / (b + c))]

print('10. General c: K and its singular values')
print('Assumption used here: b > c >= 0')
print('K_general =')
sp.pprint(K_general)
print('\nSingular values of K_general =')
sp.pprint(sv_K_general)


10. General c: K and its singular values
Assumption used here: b > c >= 0
K_general =
[   b          -c   ]
[-------  0  -------]
[ 2    2      2    2]
[b  - c      b  - c ]
[                   ]
[         1         ]
[   0     -     0   ]
[         b         ]
[                   ]
[  -c           b   ]
[-------  0  -------]
[ 2    2      2    2]
[b  - c      b  - c ]

Singular values of K_general =
   1    1    1   
[-----, -, -----]
 b - c  b  b + c 


## 11. General c: M = A^T K A

For `c != 0`, the matrix `M` is still real symmetric under the same assumptions,
so its singular values are its eigenvalues.


In [11]:
M_general = sp.simplify(A.T * K_general * A)

print('11. General c: M = A^T K A')
sp.pprint(M_general)


11. General c: M = A^T K A
[     2                                      3             ]
[b*lam                                 -c*lam              ]
[-------         0                     --------            ]
[ 2    2                                2    2             ]
[b  - c                                b  - c              ]
[                                                          ]
[                 2                    /   2     \         ]
[               mu                  mu*\lam  - mu/         ]
[   0           ---                 --------------         ]
[                b                        b                ]
[                                                          ]
[                                                         2]
[      3      /   2     \   2    4   / 2    2\ /   2     \ ]
[-c*lam    mu*\lam  - mu/  b *lam  + \b  - c /*\lam  - mu/ ]
[--------  --------------  --------------------------------]
[ 2    2         b                     / 2    2\          

## 12. General c: Characteristic Polynomial of M

A short closed form for all three singular values of `M` is not clean.
A practical exact description is: they are the three nonnegative roots of the characteristic polynomial.


In [12]:
t = sp.symbols('t', real=True)
charpoly_M_general = sp.factor(M_general.charpoly(t).as_expr())
trace_M_general = sp.factor(M_general.trace())
det_M_general = sp.factor(M_general.det())
coeffs_M_general = sp.Poly(sp.expand(M_general.charpoly(t).as_expr()), t).all_coeffs()

print('12. General c: characteristic polynomial of M')
print('charpoly_M_general(t) =')
sp.pprint(charpoly_M_general)
print('\ntrace(M_general) =')
sp.pprint(trace_M_general)
print('\ndet(M_general) =')
sp.pprint(det_M_general)
print('\nPolynomial coefficients [a3, a2, a1, a0] =')
sp.pprint(coeffs_M_general)


12. General c: characteristic polynomial of M
charpoly_M_general(t) =
 3  3      2    4  2      2    2     2    2    2  2      2   2  2      2  3    >
b *t  - 2*b *lam *t  + 2*b *lam *mu*t  - b *lam *t  - 2*b *mu *t  - b*c *t  +  >
------------------------------------------------------------------------------ >
                                                                               >

>        6          4   2            4               2   2      2    4  2      >
> 2*b*lam *t + b*lam *mu *t - 2*b*lam *mu*t + 2*b*lam *mu *t + c *lam *t  - 2* >
> ---------------------------------------------------------------------------- >
>          b*(b - c)*(b + c)                                                   >

>  2    2     2      2   2  2      6   2
> c *lam *mu*t  + 2*c *mu *t  - lam *mu 
> --------------------------------------
>                                       

trace(M_general) =
   2    4      2    2       2    2      2   2    2    4      2    2         2  >
2*b *lam  - 2*b

## 13. Process Variables

Collect the main symbolic objects for later use.


In [13]:
process_variables = {
    'A': A,
    'Sigma': Sigma,
    'Sigma_c0': Sigma_c0,
    'K': K,
    'M': M,
    'U_M': U_M,
    'S_M': S_M,
    'U_K': U_K,
    'S_K': S_K,
    'second_step_matrix': second_step_matrix,
    'second_gram': second_gram,
    'sv_step2': sv_step2,
    'W': W,
    'K_general': K_general,
    'sv_K_general': sv_K_general,
    'M_general': M_general,
    'charpoly_M_general': charpoly_M_general,
    'trace_M_general': trace_M_general,
    'det_M_general': det_M_general,
}

print('13. Process variables')
print(list(process_variables.keys()))
print('\nNote: with all 3 macro directions kept, W is a rotated basis, not a dimension reduction.')
print('For c != 0, K is simple, but M leads to a cubic equation in general.')


13. Process variables
['A', 'Sigma', 'Sigma_c0', 'K', 'M', 'U_M', 'S_M', 'U_K', 'S_K', 'second_step_matrix', 'second_gram', 'sv_step2', 'W', 'K_general', 'sv_K_general', 'M_general', 'charpoly_M_general', 'trace_M_general', 'det_M_general']

Note: with all 3 macro directions kept, W is a rotated basis, not a dimension reduction.
For c != 0, K is simple, but M leads to a cubic equation in general.
